# Fine-tuning a Sentence Transformer with InfoNCE Contrastive Loss

Companion notebook for the [Contrastive Learning](https://codemia.ai/courses/deep_learning_foundations/contrastive_learning) lesson on Codemia.

**What you will build**: `all-MiniLM-L6-v2` fine-tuned on a small domain-specific dataset of question pairs with InfoNCE loss, with retrieval quality measured before and after on a held-out test set.

**Time**: ~30-45 minutes on a T4 GPU  
**GPU required**: strongly recommended. `all-MiniLM-L6-v2` has 23M params so CPU works too, just slower.

## Setup

In [ ]:
%pip install -q torch transformers datasets matplotlib

In [ ]:
import math
import random
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import matplotlib.pyplot as plt

torch.manual_seed(0)
random.seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using {device}')

## Concept recap

A sentence transformer maps sentences to fixed-size vectors such that **semantically similar sentences end up close**. Off-the-shelf models like `all-MiniLM-L6-v2` are trained on generic web pairs; they work surprisingly well out of the box, but struggle on niche domains with unusual vocabulary.

The standard fix is contrastive fine-tuning with **InfoNCE** loss. For a batch of N pairs `(query_i, positive_i)`, we compute the cosine similarity of every query against every positive, then apply a cross-entropy where the correct class for row `i` is column `i`. Every other positive in the batch acts as an in-batch negative — we pull `q_i` toward `p_i` and push it away from everyone else. A temperature `tau` sharpens or blurs the softmax.

Our domain will be a toy corpus of programming Q&A pairs where the query and positive use very different words for the same concept (so naive lexical matching fails).

## Implementation

### Step 1 — A tiny programming Q&A dataset

We hand-craft ~30 paraphrased pairs so the notebook runs without any downloads beyond the model. The train/test split is random but disjoint in content, so the test set measures genuine generalization.

In [ ]:
# Each entry: (informal_query, formal_answer_about_same_concept).
# The two sides share almost no vocabulary — that is what makes it a fair contrastive task.
pairs = [
    ('how do i get the last item in a list', 'accessing the final element of a sequence using negative indexing'),
    ('why is my loop so slow', 'identifying algorithmic complexity bottlenecks in iteration'),
    ('what does yield do in python', 'generator functions and lazy evaluation semantics'),
    ('how to read a file line by line', 'streaming text file input with buffered reader'),
    ('stop my script from crashing on missing keys', 'safe dictionary access and KeyError handling'),
    ('what is a closure', 'functions that capture variables from enclosing lexical scope'),
    ('fastest way to sort a huge array', 'time complexity of comparison-based sorting algorithms'),
    ('how do i make my code concurrent', 'introduction to threading and multiprocessing primitives'),
    ('why are floats weird', 'IEEE 754 floating-point representation and rounding errors'),
    ('what does async await mean', 'cooperative multitasking with coroutines and event loops'),
    ('how to remove duplicates from a list', 'deduplication of sequences with hash-based sets'),
    ('my regex is eating too much memory', 'catastrophic backtracking in regular expression engines'),
    ('which is faster numpy or python loop', 'vectorization and SIMD in array computing libraries'),
    ('how do i deep copy a dict', 'recursive object duplication versus reference aliasing'),
    ('why doesnt print show up', 'standard output buffering and flush semantics'),
    ('what is a lambda', 'anonymous single-expression function literals'),
    ('how to pass by reference', 'object reference semantics in pass-by-object-reference languages'),
    ('json wont parse my string', 'JSON syntax errors and strict double-quote requirements'),
    ('how to measure how long my code takes', 'wall-clock and CPU time profiling tools'),
    ('why does += on a list mutate', 'in-place augmented assignment operators for mutable sequences'),
    ('how to turn a string into numbers', 'parsing numeric literals from string input'),
    ('what is gil', 'global interpreter lock and python thread concurrency limits'),
    ('how to write to a csv', 'comma-separated values format serialization libraries'),
    ('my import is circular', 'cyclic module dependency resolution at import time'),
    ('how to reverse a string', 'string reversal via slicing or two-pointer iteration'),
    ('what is a decorator', 'higher-order functions that wrap callables with extra behavior'),
    ('why is my model overfitting', 'generalization gap and regularization techniques'),
    ('how to download a webpage', 'HTTP client libraries for fetching remote resources'),
    ('what does pickle do', 'object serialization to binary byte streams'),
    ('how to use a gpu in pytorch', 'CUDA device placement and tensor transfer semantics'),
]

random.shuffle(pairs)
split = int(len(pairs) * 0.75)
train_pairs = pairs[:split]
test_pairs  = pairs[split:]
print(f'train pairs: {len(train_pairs)}  test pairs: {len(test_pairs)}')

### Step 2 — Load the base model and define an embed() helper

`all-MiniLM-L6-v2` is a 6-layer DistilBERT variant. We pool with **mean pooling** over non-pad tokens (that is the recipe the model was trained with) and L2-normalize the result — cosine similarity then becomes a dot product.

In [ ]:
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
print('params:', sum(p.numel() for p in model.parameters()))

In [ ]:
def mean_pool(last_hidden, attention_mask):
    # Mask out [PAD] positions before averaging — otherwise padding biases the embedding.
    mask = attention_mask.unsqueeze(-1).float()
    summed = (last_hidden * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

def embed(texts, model, batch_size=32):
    # Tokenize + forward + pool + L2-normalize. no_grad for inference use.
    outs = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            tok = tokenizer(batch, padding=True, truncation=True, max_length=64, return_tensors='pt').to(device)
            h = model(**tok).last_hidden_state
            v = mean_pool(h, tok['attention_mask'])
            v = F.normalize(v, dim=-1)  # L2-normalize so dot-product == cosine similarity
            outs.append(v)
    return torch.cat(outs, dim=0)

### Step 3 — Retrieval metric (recall@1)

Given a test set of `(query, positive)` pairs, encode every query and every positive separately, compute the NxN cosine similarity matrix, and check how often the correct positive is the top-1 for its query. Simple, and directly measures the thing we care about.

In [ ]:
def recall_at_1(model, pairs):
    queries = [q for q, _ in pairs]
    positives = [p for _, p in pairs]
    q_emb = embed(queries, model)
    p_emb = embed(positives, model)
    # sim[i, j] = cos(query_i, positive_j). The diagonal is the correct pair.
    sim = q_emb @ p_emb.T
    top1 = sim.argmax(dim=1)
    correct = (top1 == torch.arange(len(pairs), device=device)).float().mean().item()
    return correct, sim.cpu()

r1_before, sim_before = recall_at_1(model, test_pairs)
print(f'test recall@1 BEFORE fine-tuning: {r1_before:.3f}')

### Step 4 — Forward with gradient, for training

In [ ]:
def encode_train(texts):
    # Same as embed() but *keeps* gradients. Used during the training loop only.
    tok = tokenizer(texts, padding=True, truncation=True, max_length=64, return_tensors='pt').to(device)
    h = model(**tok).last_hidden_state
    v = mean_pool(h, tok['attention_mask'])
    return F.normalize(v, dim=-1)

### Step 5 — InfoNCE loss

Symmetric InfoNCE: treat `(query -> positive)` and `(positive -> query)` as two directions and average. This is what CLIP and most modern retrieval models use.

In [ ]:
def info_nce(q, p, temperature=0.05):
    # q and p are already L2-normalized, so q @ p.T is cosine similarity in [-1, 1].
    logits = (q @ p.T) / temperature
    # Correct class for row i is column i — the diagonal. In-batch negatives come for free.
    labels = torch.arange(q.size(0), device=q.device)
    # Symmetric: once with q as query, once with p as query.
    return 0.5 * (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels))

## Experiment: fine-tune

In [ ]:
BATCH_SIZE = 8
EPOCHS = 10
LR = 2e-5   # small LR because we are adapting a pretrained model — big LR destroys it

opt = torch.optim.AdamW(model.parameters(), lr=LR)
history = {'train_loss': [], 'test_r1': [r1_before]}

model.train()
for epoch in range(EPOCHS):
    random.shuffle(train_pairs)
    epoch_losses = []
    for start in range(0, len(train_pairs), BATCH_SIZE):
        batch = train_pairs[start:start + BATCH_SIZE]
        if len(batch) < 2:
            continue  # InfoNCE needs at least 2 pairs so there are negatives
        q_batch = [q for q, _ in batch]
        p_batch = [p for _, p in batch]
        q = encode_train(q_batch)
        p = encode_train(p_batch)
        loss = info_nce(q, p, temperature=0.05)
        opt.zero_grad()
        loss.backward()
        opt.step()
        epoch_losses.append(loss.item())
    r1, _ = recall_at_1(model, test_pairs)
    avg = sum(epoch_losses) / max(1, len(epoch_losses))
    history['train_loss'].append(avg)
    history['test_r1'].append(r1)
    print(f'epoch {epoch+1:2d}  loss={avg:.4f}  test recall@1={r1:.3f}')

In [ ]:
r1_after, sim_after = recall_at_1(model, test_pairs)
print(f'test recall@1 BEFORE: {r1_before:.3f}')
print(f'test recall@1 AFTER : {r1_after:.3f}')
print(f'delta: {r1_after - r1_before:+.3f}')

In [ ]:
# Two plots: loss curve, and recall@1 over time (including the initial point before training).
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(history['train_loss'], 'o-')
ax[0].set_title('InfoNCE loss'); ax[0].set_xlabel('epoch'); ax[0].grid(alpha=0.3)
ax[1].plot(history['test_r1'], 'o-')
ax[1].set_title('test recall@1'); ax[1].set_xlabel('epoch'); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

### Visualize the similarity matrices

The diagonal should light up after training — that is literally what the loss pushes for. Brighter off-diagonals mean the model is confusing unrelated queries.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(sim_before, vmin=-0.2, vmax=1.0, cmap='magma')
ax[0].set_title('before')
im = ax[1].imshow(sim_after, vmin=-0.2, vmax=1.0, cmap='magma')
ax[1].set_title('after')
fig.colorbar(im, ax=ax.ravel().tolist(), label='cos sim')
plt.show()

## Variations

### Temperature sweep

InfoNCE's temperature matters. Too high and the softmax flattens (no learning signal); too low and it spikes on whichever negative happens to be closest at init (unstable). Sweep it and pick the best.

In [ ]:
# Rerun training from a fresh model for each temperature. Expensive-ish but illuminating.
def fresh_model():
    return AutoModel.from_pretrained(MODEL_NAME).to(device)

def train_and_eval(temp, epochs=6):
    global model
    model = fresh_model()
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    model.train()
    for _ in range(epochs):
        random.shuffle(train_pairs)
        for start in range(0, len(train_pairs), BATCH_SIZE):
            batch = train_pairs[start:start + BATCH_SIZE]
            if len(batch) < 2: continue
            q = encode_train([x for x, _ in batch])
            p = encode_train([x for _, x in batch])
            loss = info_nce(q, p, temperature=temp)
            opt.zero_grad(); loss.backward(); opt.step()
    r1, _ = recall_at_1(model, test_pairs)
    return r1

temps = [0.01, 0.05, 0.1, 0.3, 1.0]
sweep = []
for t in temps:
    r = train_and_eval(t)
    print(f'  temperature={t:.2f}  recall@1={r:.3f}')
    sweep.append(r)

plt.figure(figsize=(6, 3.5))
plt.plot(temps, sweep, 'o-')
plt.xscale('log')
plt.xlabel('temperature'); plt.ylabel('test recall@1')
plt.title('Temperature sweep'); plt.grid(alpha=0.3); plt.show()

## What to try next

- Plug in a real domain dataset (e.g. MS MARCO, BEIR, or scrape your own FAQ corpus) and train for a proper number of steps.
- Add **hard negative mining**: after a warmup, for each query sample the top-k most similar *wrong* answers as extra negatives. This is the trick every production retrieval system uses.
- Compare against a baseline BM25 (classic lexical retrieval). On in-domain data the fine-tuned encoder should win decisively.
- Freeze all layers except the last 2 and see how much quality you keep — this is the cheap-fine-tune setup used for many real deployments.